## Training an Audio Latent Diffusion Transformer (DiT)  

This notebook uses the [PLaTune-mini](#) repository to train an audio generative model on a small guitar dataset: [GuitarSet](https://zenodo.org/records/3371780).

PLaTune (Pretrained Latents Tuner) (Nabi et al., 2025) is a Latent Diffusion Transformer for generating audio data with the option to add musical controls (e.g., melody, dynamics, instrument types). It implements a technique for latent diffusion called [Rectified Flow](https://rectifiedflow.github.io/),

PLaTune-mini is a toy version of PLaTune, with a smaller number of model parameters, and an additional data augmentation technique to allow models to **train on smaller datasets**.



**Original research paper of PLaTune:**  
> Nabi, S., Demerlé, N., Peeters, G., Bevilacqua, F., Esling, P., 2025. Adding temporal musical controls on top of pretrained generative models, in: Proceedings of the 26th International Society for Music Information Retrieval Conference. ISMIR, pp. 793–800. https://doi.org/10.5281/zenodo.17811482

## 1. Config you environment

In [ ]:
!git clone https://github.com/anonymous-capybara/platune-mini.git

In [ ]:
import sys
!pip install -r requirements.txt

if sys.platform == 'darwin':
    # MacOS
    !pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
else:
    # Windows/Linux with CUDA
    !pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
    !pip install tensorboard>=2.20.0 setuptools==81.0.0 --force-reinstall

## Prepare Dataset

### Extract latents from Guitarset, using a pretrained autoencoder

In [ ]:
!python prepare_dataset.py \
    --output_path datasets/guitatset-test \
    --input_path "/Users/jasperrr/PhD/26-01-fourier/guitarset-mono-mic-small" \
    --config simple \
    --emb_model_path music2latent \
    --parser_name simple_parser \
    --gpu 0 \
    --num_signal 131072 \
    --cut_silences \
    --save_waveform \
    --use_basic_pitch --midi_attributes \
    -l rms -l integrated_loudness -l loudness1s \
    --val_num_chunks 100

In [ ]:
!python prepare_augmented_dataset_v4.py \
        --lmdb_path datasets/guitatset \
        --output_path datasets/guitatset_aug \
        --n_augmented 200 \
        --db_size 4 \
        --config augmentations_v4 \
        --codec music2latent \
        --gpu 0

## Train PLaTune-mini

### 1. Import the training pieces

This cell imports the mini model, the dataset loader, and a couple of small helper functions. The helpers are only here to keep the actual training cells easy to read.

In [ ]:
import json
import os


import pytorch_lightning as pl
import torch

from platune.datasets.dataset import load_data
from platune.model_mini import PLaTuneMini

torch.set_float32_matmul_precision("high")
from platune.utils import search_for_run, select_accelerator


### 2. Choose the run settings

This is the main cell you are likely to edit. It defines where the training and validation LMDBs live, where checkpoints will be saved, and a few core training hyperparameters.

In [ ]:
db_path = "training_data/guitatset"
val_lmdb_path = "training_data/guitatset_val"
name = "guitatset_mini"
save_path = "runs"

gpu = 1
ckpt = None  # e.g. "runs/guitatset_mini/version_0/checkpoints/last.ckpt"

max_steps = 300_000
val_every = 15_000
lmdb_keys_filename = "lmdb_keys"

batch_size = 32
n_workers = 7
lr = 1e-4
nb_steps = 20
n_audio_examples = 12


### 3. Load the unconditional latent dataset

PLaTune-mini is unconditional, so we pass empty control lists into the existing dataset loader. The loader still returns the same tuple shape as the full model pipeline, but the mini model only uses the latent tensor `z`.

In [ ]:
train_loader, val_loader, _ = load_data(
    data_path=db_path,
    discrete_keys=[],
    continuous_keys=[],
    batch_size=batch_size,
    n_workers=n_workers,
    lmdb_keys_file=lmdb_keys_filename,
    val_data_path=val_lmdb_path,
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")


#### 3.1 Visualise some latents in the dataset

In [ ]:
import matplotlib.pyplot as plt

batch = next(iter(train_loader))

z = batch[0].detach().cpu()

print(f"latent batch shape: {tuple(z.shape)}")
print(f"min: {z.min().item():.4f}, max: {z.max().item():.4f}")

# Visualize the first sample in the batch
sample = z[0]  # shape: (latent_dim, time)
sample_2d = sample.reshape(sample.shape[0], -1)
plt.figure(figsize=(10, 4))
plt.imshow(sample_2d.numpy(), aspect="auto", origin="lower", cmap="magma")
plt.xlabel("Time")
plt.ylabel("Latent Dimension")

plt.tight_layout()
plt.show()

### 4. Build the mini model and save the run config

Here we start from the default PLaTune-mini settings, apply a few notebook overrides, and save them to `config.json` so the run can be reproduced later.  

In [ ]:
model_config = PLaTuneMini.default_model_config()
model_config.update({
    "lr": lr,
    "nb_steps": nb_steps,
    "n_audio_examples": n_audio_examples,
})
model = PLaTuneMini(**model_config)

run_dir = os.path.join(save_path, name)
os.makedirs(run_dir, exist_ok=True)

config = {
    "cli_args": {
        "db_path": db_path,
        "val_lmdb_path": val_lmdb_path,
        "name": name,
        "save_path": save_path,
        "gpu": gpu,
        "ckpt": ckpt,
        "max_steps": max_steps,
        "val_every": val_every,
        "lmdb_keys_filename": lmdb_keys_filename,
        "batch_size": batch_size,
        "n_workers": n_workers,
        "lr": lr,
        "nb_steps": nb_steps,
        "n_audio_examples": n_audio_examples,
    },
    "model_config": model_config,
}

with open(os.path.join(run_dir, "config.json"), "w") as config_out:
    json.dump(config, config_out, indent=2)

config


### 5. Create the Lightning trainer

This cell sets up checkpointing, picks the device, and decides how often validation should run. Validation logs synthesized audio samples so you can hear how the diffusion model evolves during training.

In [ ]:
callbacks = [pl.callbacks.ModelCheckpoint(filename="last")]

val_check = {}
if val_loader is not None and len(train_loader) > 0:
    if len(train_loader) >= val_every:
        val_check["val_check_interval"] = val_every
        print(f"Validation will be checked every {val_every} training steps.")
    else:
        n_epoch = max(1, val_every // len(train_loader))
        val_check["check_val_every_n_epoch"] = n_epoch
        print(f"Validation will be checked every {n_epoch} epochs.")

accelerator, devices = select_accelerator(gpu)
print(f"device - selected gpu: {accelerator}:{devices}")

trainer = pl.Trainer(
    logger=pl.loggers.TensorBoardLogger(save_path, name=name),
    accelerator=accelerator,
    devices=devices,
    callbacks=callbacks,
    max_epochs=100000,
    max_steps=max_steps,
    profiler="simple",
    enable_progress_bar=True,
    **val_check,
)


### 6. Start training

If `ckpt` points to a previous checkpoint, training resumes from there. Otherwise this starts a fresh unconditional PLaTune-mini run.

In [ ]:
run = search_for_run(ckpt)
if run is not None:
    print(f"Restarting from checkpoint: {run}")

trainer.fit(model, train_loader, val_loader, ckpt_path=run)


## Tasks